# Agona student LoRA

Small QLoRA fine-tune of a real open student, then the same holdout check before and after.

- Student: `Qwen/Qwen2.5-1.5B-Instruct`
- License: Apache 2.0 (`apache-2.0`), from the Hugging Face model card `cardData.license`, checked 2026-11-12. License file: https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/blob/main/LICENSE
- Input: an approved JSONL file from the Agona site. Each line is `{"messages":[{"role":"user",...},{"role":"assistant",...}]}`.
- The holdout rows are chosen with a fixed seed and are never used for training.
- Output: a LoRA adapter and `agona-metrics.json`. The site can import that file. This notebook does not invent scores.

Turn on a GPU. Internet is needed the first time so the student weights can download.


## 1. Packages

Kaggle GPU images often already have these. This cell installs them if the import fails.


In [ ]:
import importlib.util
import subprocess
import sys

needed = ["transformers", "peft", "trl", "bitsandbytes", "accelerate", "datasets"]
missing = [name for name in needed if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("packages ready")


## 2. Settings

Put the approved JSONL on Kaggle as a dataset, or set `AGONA_JSONL` to its path. The usual Kaggle path is `/kaggle/input/agona-approved/agona-approved.jsonl`.


In [ ]:
import glob
import os

STUDENT_ID = "Qwen/Qwen2.5-1.5B-Instruct"
STUDENT_LICENSE = "apache-2.0"
SEED = 2108
HOLDOUT_FRACTION = 0.2
MIN_ROWS = 8
MAX_LEN = 512
MAX_NEW_TOKENS = 64
LORA_R = 8
LORA_ALPHA = 16

def output_dir():
    if os.path.isdir("/kaggle/working"):
        return "/kaggle/working"
    path = os.path.abspath("agona-output")
    os.makedirs(path, exist_ok=True)
    return path

def find_jsonl():
    chosen = os.environ.get("AGONA_JSONL", "").strip()
    candidates = [chosen] if chosen else []
    candidates += [
        "/kaggle/input/agona-approved/agona-approved.jsonl",
        "/kaggle/input/agona-approved/approved.jsonl",
    ]
    candidates += sorted(glob.glob("/kaggle/input/**/*.jsonl", recursive=True))
    for path in candidates:
        if path and os.path.isfile(path):
            return path
    raise FileNotFoundError(
        "No approved JSONL found. Upload agona-approved.jsonl as a Kaggle dataset, "
        "or set AGONA_JSONL to the file path."
    )

OUT = output_dir()
DATA_PATH = find_jsonl()
print("student", STUDENT_ID)
print("license", STUDENT_LICENSE)
print("data", DATA_PATH)
print("output", OUT)


## 3. Load examples and lock the holdout

The same seed always picks the same holdout. Those rows are not in the training set.


In [ ]:
import json
import random

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            messages = item.get("messages")
            if not isinstance(messages, list) or len(messages) < 2:
                raise ValueError(f"Line {line_no} needs a messages list with a user and an assistant turn.")
            user = next((m for m in messages if m.get("role") == "user" and m.get("content")), None)
            assistant = next((m for m in messages if m.get("role") == "assistant" and m.get("content")), None)
            if not user or not assistant:
                raise ValueError(f"Line {line_no} needs user and assistant text.")
            rows.append({
                "messages": [
                    {"role": "user", "content": str(user["content"])},
                    {"role": "assistant", "content": str(assistant["content"])},
                ]
            })
    return rows

def split_holdout(rows):
    ordered = list(rows)
    random.Random(SEED).shuffle(ordered)
    hold_n = max(1, int(round(len(ordered) * HOLDOUT_FRACTION)))
    hold_n = min(hold_n, len(ordered) - 1)
    holdout = ordered[:hold_n]
    train = ordered[hold_n:]
    hold_ids = {json.dumps(row, sort_keys=True) for row in holdout}
    train_ids = {json.dumps(row, sort_keys=True) for row in train}
    if hold_ids & train_ids:
        raise RuntimeError("A holdout row also landed in training. Stop.")
    return train, holdout

rows = load_jsonl(DATA_PATH)
if len(rows) < MIN_ROWS:
    raise SystemExit(f"Need at least {MIN_ROWS} approved examples so a holdout can stay out of training. Found {len(rows)}.")
train_rows, holdout_rows = split_holdout(rows)
print(f"train {len(train_rows)}  holdout {len(holdout_rows)}")


## 4. Score a model on the holdout

Exact match and token overlap use the same short greedy answers. Loss is the model's own loss on the assistant words. Holdout rows are not trained on.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def normalize(text):
    return " ".join(str(text or "").lower().split())

def token_f1(pred, gold):
    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()
    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0
    from collections import Counter
    overlap = sum((Counter(pred_tokens) & Counter(gold_tokens)).values())
    precision = overlap / len(pred_tokens)
    recall = overlap / len(gold_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def load_student():
    tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID, trust_remote_code=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        STUDENT_ID,
        quantization_config=quant,
        device_map="auto",
        trust_remote_code=False,
    )
    model.config.use_cache = False
    return model, tokenizer

def chat_prompt(tokenizer, user_text):
    messages = [{"role": "user", "content": user_text}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.inference_mode()
def generate_answer(model, tokenizer, user_text):
    prompt = chat_prompt(tokenizer, user_text)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

def holdout_loss(model, tokenizer, rows):
    model.eval()
    losses = []
    for row in rows:
        prompt = chat_prompt(tokenizer, row["messages"][0]["content"])
        full = tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)
        prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
        full_ids = tokenizer(full, add_special_tokens=False, truncation=True, max_length=MAX_LEN).input_ids
        if len(full_ids) <= len(prompt_ids):
            continue
        input_ids = torch.tensor([full_ids], device=model.device)
        labels = input_ids.clone()
        labels[0, :len(prompt_ids)] = -100
        out = model(input_ids=input_ids, labels=labels)
        losses.append(float(out.loss.detach().cpu()))
    if not losses:
        return None
    return sum(losses) / len(losses)

def score_model(model, tokenizer, rows):
    model.eval()
    exact_hits = []
    overlaps = []
    notes = []
    for row in rows:
        gold = row["messages"][1]["content"]
        try:
            pred = generate_answer(model, tokenizer, row["messages"][0]["content"])
        except Exception as err:
            notes.append(f"One holdout answer could not be generated: {err}")
            continue
        exact_hits.append(1.0 if normalize(pred) == normalize(gold) else 0.0)
        overlaps.append(token_f1(pred, gold))
    result = {}
    if exact_hits:
        result["exact"] = sum(exact_hits) / len(exact_hits)
    if overlaps:
        result["token_f1"] = sum(overlaps) / len(overlaps)
    loss = holdout_loss(model, tokenizer, rows)
    if loss is not None:
        result["loss"] = loss
    return result, notes

print("scoring helpers ready")


## 5. Before: the student with no adapter


In [ ]:
base_model, tokenizer = load_student()
baseline, baseline_notes = score_model(base_model, tokenizer, holdout_rows)
print("before", json.dumps(baseline))
del base_model
torch.cuda.empty_cache()


## 6. QLoRA supervised fine-tune

Only the training rows are used. Rank 8, one pass, short sequences, so a free GPU can finish.


In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from datasets import Dataset

model, tokenizer = load_student()
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

def to_text(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_list([to_text(row) for row in train_rows])
use_bf16 = torch.cuda.is_bf16_supported()
train_args = dict(
    output_dir=os.path.join(OUT, "agona-train"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=5,
    save_strategy="no",
    bf16=use_bf16,
    fp16=not use_bf16,
    report_to="none",
    max_length=MAX_LEN,
    dataset_text_field="text",
    packing=False,
)

trainer = None
errors = []
try:
    from trl import SFTConfig, SFTTrainer
    trainer = SFTTrainer(
        model=model,
        args=SFTConfig(**train_args),
        train_dataset=train_ds,
        peft_config=lora,
        processing_class=tokenizer,
    )
except Exception as err:
    errors.append(str(err))
    from trl import SFTTrainer
    from transformers import TrainingArguments
    args = TrainingArguments(
        output_dir=train_args["output_dir"],
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        num_train_epochs=1,
        logging_steps=5,
        save_strategy="no",
        bf16=use_bf16,
        fp16=not use_bf16,
        report_to="none",
    )
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        args=args,
        train_dataset=train_ds,
        peft_config=lora,
        dataset_text_field="text",
        max_seq_length=MAX_LEN,
    )

trainer.train()
adapter_dir = os.path.join(OUT, "agona-student-lora")
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("adapter", adapter_dir)


## 7. After: the same holdout, with the adapter


In [ ]:
trained, trained_notes = score_model(trainer.model, tokenizer, holdout_rows)
print("after", json.dumps(trained))

from datetime import datetime, timezone
metrics = {
    "schema": "agona-metrics-v1",
    "student": STUDENT_ID,
    "license": STUDENT_LICENSE,
    "method": "qlora-sft",
    "created": datetime.now(timezone.utc).isoformat(),
    "holdout_size": len(holdout_rows),
    "train_size": len(train_rows),
    "seed": SEED,
    "baseline": baseline,
    "trained": trained,
    "notes": baseline_notes + trained_notes + errors,
}
metrics_path = os.path.join(OUT, "agona-metrics.json")
with open(metrics_path, "w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)
    handle.write("\n")
print("metrics", metrics_path)
print(json.dumps(metrics, indent=2))


## 8. Bring the numbers back

Download `agona-metrics.json` from the notebook output. On the Agona Scores page, import that file. The page shows only numbers that are in the file.
